k-means clustering

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def find_closest_centroids(X, centroids):
    m, n = X.shape
    K = centroids.shape[0]
    idx = np.zeros(m, dtype=int)

    for i in range(m):
        distances = []

        for j in range(K):
            distance = np.linalg.norm(X[i]-centroids[j])
            distances.append(distance)
        idx[i] = np.argmin(distances)
        
    return idx

In [ ]:
X = np.load("ex7_X (1).npy")
initial_centroids = np.array([[3,3], [6,2],[8,5]])
print(X[:5])
print(X.shape)

In [ ]:
idx = find_closest_centroids(X, initial_centroids)
idx

In [ ]:
def compute_centroids(X, idx, K):
    m,n = X.shape 
    centroids = np.zeros((K,n))

    for k in range(K):
        points = X[idx==k]
        X_sum = np.sum(points, axis=0)
        centroids[k] = X_sum/len(points)
    
    return centroids


In [ ]:
centroids = compute_centroids(X, idx, K=3)
centroids

In [ ]:
def initial_centroid(X, K):
    m, n = X.shape 

    randidx = np.random.permutation(m)
    centroids = X[randidx[:K]]
    return centroids


In [ ]:
ini_centroids = initial_centroid(X, K=3)
ini_centroids 

In [ ]:
def compute_cost(X, centroids, idx):
    m, n = X.shape
    cost = 0
    for i in range(m):
        centroid = centroids[idx[i]]
        cost += np.linalg.norm(X[i]-centroid)**2
    
    return cost/m


In [ ]:
compute_cost(X, ini_centroids, idx)

In [ ]:
def kmeans_algorithm(X, initial_centroids, max_iter):
    m,n = X.shape
    K = initial_centroids.shape[0]
    centroids = initial_centroids.copy()
    cost_list = []
    idx = np.zeros(m, dtype=int)
    for i in range(max_iter):
        idx = find_closest_centroids(X, centroids)
        centroids = compute_centroids(X, idx, K)
        cost = compute_cost(X, centroids, idx)
        cost_list.append(cost)

        if i in [0, 1, 2, max_iter-1]:
            fig, ax = plt.subplots(1,1, figsize=(8,5))
            ax.scatter(X[:,0], X[:,1], c = idx)
            ax.scatter(centroids[:,0], centroids[:,1], marker="X",c="red", s=200)
            ax.set_title(f"iteration {i}")
            plt.show()

    plt.plot(range(len(cost_list)), cost_list)
    plt.xlabel("iteration")
    plt.ylabel("cost")
    plt.show()

    return idx, centroids


In [ ]:
initial_centroids = initial_centroid(X, K=3)
idx, centroids = kmeans_algorithm(X, initial_centroids, max_iter=10)

Image compression

In [ ]:
original_image = plt.imread("bird_small.png")

In [ ]:
plt.imshow(original_image)

In [ ]:
original_image.shape

In [ ]:
X_image_reshape = np.reshape(original_image, (16384, 3))

In [ ]:
initial_centroids = initial_centroid(X_image_reshape, K=16)
idx, centroids = kmeans_algorithm(X_image_reshape, initial_centroids, max_iter=10)

In [ ]:
X_recovered = centroids[idx, :]
X_recovered = np.reshape(X_recovered, original_image.shape)

In [ ]:
plt.imshow(original_image)
plt.title("original image")
plt.show()

plt.imshow(X_recovered)
plt.title("image compression")
plt.show()

Anomaly Detection

In [ ]:
X_train = np.load("X_part1.npy")
X_val = np.load("X_val_part1.npy")
y_val = np.load("y_val_part1.npy")
print(X_train.shape)
print(X_val.shape)
print(y_val.shape)

In [ ]:
def multivariate_gaussian(X, mu, var):
    """
    Computes the probability 
    density function of the examples X under the multivariate gaussian 
    distribution with parameters mu and var. If var is a matrix, it is
    treated as the covariance matrix. If var is a vector, it is treated
    as the var values of the variances in each dimension (a diagonal
    covariance matrix
    """
    
    k = len(mu)
    
    if var.ndim == 1:
        var = np.diag(var)
        
    X = X - mu
    p = (2* np.pi)**(-k/2) * np.linalg.det(var)**(-0.5) * \
        np.exp(-0.5 * np.sum(np.matmul(X, np.linalg.pinv(var)) * X, axis=1))
    
    return p

In [ ]:
def estimate_gaussian(X):
    mu = np.mean(X, axis=0)
    var = np.var(X, axis=0)

    return mu, var

In [ ]:
def select_epsilon(y_val, p_val):

    best_F1 = 0
    best_epsilon = 0
    F1 = 0

    step_size = (max(p_val) - min(p_val))/1000
    for epsilon in np.arange(min(p_val), max(p_val), step_size):

        predictions = (p_val<epsilon)
        tp = np.sum((predictions==1)&(y_val==1))
        fp = np.sum((predictions==1)&(y_val==0))
        fn = np.sum((predictions==0)&(y_val==1))

        precision = tp/(tp+fp) if (tp+fp) > 0 else 0
        recall = tp/(tp+fn) if (tp+fn) > 0 else 0

        F1 = 0 if (precision + recall) == 0 else \
        (2 * precision * recall) / (precision + recall)

        if F1>best_F1:
            best_F1 = F1
            best_epsilon = epsilon
        
    return best_F1, best_epsilon

In [ ]:
mu, var = estimate_gaussian(X_train)
p_val = multivariate_gaussian(X_val, mu, var)

F1, epsilon = select_epsilon(y_val, p_val)

In [ ]:
F1

In [ ]:
epsilon

In [ ]:
plt.scatter(X_train[:,0], X_train[:,1], marker="x")
plt.axis([0,30, 0, 30])
plt.show()